# Search Enamine with DeepMedChem and RDKit

This notebook searches the hosted Enamine REAL database with the `deepmedchem` Python client, then uses RDKit to draw the query and returned molecules. RDKit is only used for local visualization; the search runs through the DeepMedChem API.

## Setup

Install the SDK from the repository root and add the notebook-only dependencies:

```bash
pip install -e .
pip install jupyterlab pandas rdkit
```

In [ ]:
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw

from deepmedchem import Client

## Define and draw the query

Replace `...` with your DeepMedChem API key. For shared notebooks, prefer loading the key from an environment variable instead of saving it in the notebook.

In [ ]:
API_KEY = "..."
DATABASE = "enamine-real-v5a"
QUERY_NAME = "Aspirin"
QUERY_SMILES = "CC(=O)OC1=CC=CC=C1C(=O)O"

query_mol = Chem.MolFromSmiles(QUERY_SMILES)
Draw.MolToImage(query_mol, size=(400, 280), legend=QUERY_NAME)

## Run the search

In [ ]:
dmc = Client(api_key=API_KEY)
result = dmc.search(QUERY_SMILES, database=DATABASE, limit=9)
dmc.close()

print(f"database={result.database_id} release={result.database_release}")
print(f"returned {len(result.results)} molecules")

A representative response begins like this; releases and rankings can change:

```text
database=enamine-real-v5a release=2026-08-29.1
returned 9 molecules
```

## Inspect the result table

In [ ]:
rows = [
    {
        "rank": hit.get("rank"),
        "score": hit.get("score"),
        "product_id": hit.get("product_id"),
        "smiles": hit.get("smiles"),
    }
    for hit in result.results
]
results_df = pd.DataFrame(rows)
display(results_df)

## Draw the result grid

Each legend contains the API rank, similarity score, and product ID. Invalid structures, if any, are skipped before drawing.

In [ ]:
molecules = []
legends = []

for hit in result.results:
    molecule = Chem.MolFromSmiles(hit["smiles"])
    if molecule is None:
        continue
    molecules.append(molecule)
    legends.append(
        f"Rank {hit['rank']} | score {hit['score']:.4f}\n{hit['product_id']}"
    )

Draw.MolsToGridImage(
    molecules,
    molsPerRow=3,
    subImgSize=(320, 260),
    legends=legends,
    useSVG=True,
)

## Optional: highlight the exact query

Similarity and exact identity are different concepts. Morgan fingerprints can assign the same score to stereochemical variants, so compare canonical isomeric SMILES when exact identity matters.

In [ ]:
canonical_query = Chem.MolToSmiles(query_mol, isomericSmiles=True)
results_df["exact_query"] = [
    Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True)
    == canonical_query
    for smiles in results_df["smiles"]
]
display(results_df)

## Search with SMILES and SMARTS substructures

The junction example is a concrete molecular graph, so it is submitted as `smiles`. The remaining examples use genuine SMARTS atom properties, alternatives, charges, or recursive expressions and are submitted as `smarts`. Complex recursive SMARTS can require a longer timeout.

In [ ]:
SUBSTRUCTURE_QUERIES = {
    "junction urea ring (SMILES)": ("smiles", "CNC(=O)N1CCC1"),
    "acyclic hydrazide (SMARTS)": ("smarts", "[N;R0][N;R0]C(=O)"),
    "acyclic urea (SMARTS)": ("smarts", "[N;R0]C(=O)[N;R0]"),
    "protic amide or acid (SMARTS)": ("smarts", "[O,N;H1]C(=O)"),
}

with Client(api_key=API_KEY) as dmc:
    substructure_results = {
        name: dmc.search_substructure(
            pattern,
            query_format=query_format,
            database=DATABASE,
            limit=9,
            timeout_seconds=60,
        )
        for name, (query_format, pattern) in SUBSTRUCTURE_QUERIES.items()
    }

pd.DataFrame([
    {"query": name, "returned": len(search.results), "timed_out": search.timed_out}
    for name, search in substructure_results.items()
])

### Optional advanced recursive SMARTS

The SQC amino-acid query below demonstrates recursive SMARTS. It is valid, but can require a full-space scan and may time out on Enamine, so it is not run automatically with the interactive examples.

In [ ]:
AMINO_ACID_SMARTS = (
    "[$([NH2]),$([NH][c,CX4]),$(N([c,CX4])[c,CX4]);!$(NC=O)][CX4]C(=O)[OH]"
)

# Uncomment to run the advanced query with the API's maximum timeout.
# with Client(api_key=API_KEY) as dmc:
#     amino_acids = dmc.search_substructure(
#         AMINO_ACID_SMARTS, query_format="smarts", database=DATABASE,
#         limit=9, timeout_seconds=120,
#     )